# CRZ2025 Shock Data

## Getting R007 (National Interbank Funding Center)

In [2]:
import os
import akshare as ak
import pandas as pd
import time
import numpy as np
from functools import reduce


os.chdir('/Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data')
print("Current Directory:", os.getcwd())


Current Directory: /Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data


In [2]:
# 1. Create a list of years to loop through
years = range(2000, 2026) 
all_data = []

print("Starting download loop...")

for year in years:
    start_date = f"{year}0101"
    end_date = f"{year}1231"
    
    print(f"Fetching data for {year}...")
    
    try:
        # Fetch data for the specific year
        df = ak.repo_rate_hist(start_date=start_date, end_date=end_date)
        
        # Optional: Add a 'Year' column if you want to track it explicitly
        # df['Year'] = year 
        
        all_data.append(df)
        
        # Pause briefly to be polite to the server (prevents blocking)
        time.sleep(1) 
        
    except Exception as e:
        print(f"Error fetching data for {year}: {e}")
        # Continue to the next year even if one fails

# 2. Combine all years into one DataFrame
if all_data:
    full_repo_df = pd.concat(all_data, ignore_index=True)
    
    # 3. Clean duplicates (just in case) and Sort
    # The date column name might vary slightly; usually it is 'date' or '日期'
    # Check the column names if sort fails.
    if '日期' in full_repo_df.columns:
        full_repo_df.rename(columns={'日期': 'date'}, inplace=True)
        
    full_repo_df['date'] = pd.to_datetime(full_repo_df['date'])
    full_repo_df.sort_values('date', inplace=True)
    
    # 4. Filter for R007 specifically
    # The column for the rate might be '收盘' (Close) or similar.
    # Often the dataframe contains multiple repo types. 
    # If the output contains a "symbol" or "code" column, filter for "R-007" or similar.
    # (Inspect full_repo_df.head() to be sure of the column name for 7-day repo)

    print("Download Complete!")
    print(full_repo_df.head())
    print(f"Total rows: {len(full_repo_df)}")
    

else:
    print("No data fetched.")


Starting download loop...
Fetching data for 2000...
Fetching data for 2001...
Fetching data for 2002...
Fetching data for 2003...
Fetching data for 2004...
Fetching data for 2005...
Fetching data for 2006...
Fetching data for 2007...
Fetching data for 2008...
Fetching data for 2009...
Fetching data for 2010...
Fetching data for 2011...
Fetching data for 2012...
Fetching data for 2013...
Fetching data for 2014...
Fetching data for 2015...
Fetching data for 2016...
Fetching data for 2017...
Fetching data for 2018...
Fetching data for 2019...
Fetching data for 2020...
Fetching data for 2021...
Fetching data for 2022...
Fetching data for 2023...
Fetching data for 2024...
Fetching data for 2025...
Download Complete!
        date  FR001  FR007  FR014  FDR001  FDR007  FDR014
0 2000-01-04    NaN   2.57    NaN     NaN     NaN     NaN
1 2000-01-05    NaN   2.56    NaN     NaN     NaN     NaN
2 2000-01-06    NaN   2.57    NaN     NaN     NaN     NaN
3 2000-01-07    NaN   2.58    NaN     NaN     N

In [3]:
full_repo_df['date'] = pd.to_datetime(full_repo_df['date'])


In [4]:
full_repo_df['quarter'] = full_repo_df['date'].dt.to_period('Q')
china_r007_df = full_repo_df.groupby('quarter')['FR007'].mean().reset_index()
china_r007_df['quarter'] = china_r007_df['quarter'].astype(str)

print(china_r007_df.head())

  quarter     FR007
0  2000Q1  2.506311
1  2000Q2  2.398145
2  2000Q3  2.332576
3  2000Q4  2.360902
4  2001Q1  2.501951


## Getting CPI (Jin10)

In [5]:
cpi_df = ak.macro_china_cpi_yearly()


In [6]:
# Remove unnecessary columns and rename to English
cpi_df = cpi_df[["日期", "今值"]].rename(columns={"日期": "date", "今值": "cpi"})


In [7]:
# Convert date into proper format
cpi_df['date'] = pd.to_datetime(cpi_df['date'])

In [8]:
# Remove values before 2000
cpi_df = cpi_df[cpi_df['date'].dt.year >= 2000]

In [9]:
# Convert into quarter format
cpi_df['quarter'] = cpi_df['date'].dt.to_period('Q')

In [10]:
cpi_df = cpi_df.groupby('quarter')['cpi'].mean().reset_index()

## Cleaning Real GDP (National Bureau of Statistics)

In [19]:
gdp_df = pd.read_excel("China_RealGDP.xlsx")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [20]:
gdp_df = gdp_df[["指标名称", "中国:GDP:不变价:当季同比"]].rename(columns={"指标名称": "date", "中国:GDP:不变价:当季同比": "realgdpgrowth"})

In [ ]:
# Remove the notes rows
gdp_df = gdp_df.iloc[4:-2]

In [22]:
gdp_df['date'] = pd.to_datetime(gdp_df['date']).dt.to_period('Q')

In [ ]:
gdp_df = gdp_df[gdp_df['date'].dt.year >= 2000]
gdp_df['realgdpgrowth'] = pd.to_numeric(gdp_df['realgdpgrowth'], errors='coerce')

In [25]:
gdp_df = gdp_df.rename(columns={'date': 'quarter'})

## Combining all three data

In [29]:
china_r007_df['quarter'] = china_r007_df['quarter'].astype(str)
cpi_df['quarter'] = cpi_df['quarter'].astype(str)
gdp_df['quarter'] = gdp_df['quarter'].astype(str)

In [30]:
# 1. Put dataframes in a list
df_crz2025 = [china_r007_df, cpi_df, gdp_df]

# 2. Define a merging function (Inner Join = Intersection of dates)
# This keeps only rows where 'quarter' exists in ALL dataframes
df_crz2025 = reduce(lambda left, right: pd.merge(left, right, on='quarter', how='inner'), df_crz2025)

# 3. specific cleanup
# Set quarter as the index and sort it chronologically
df_crz2025 = df_crz2025.set_index('quarter').sort_index()


In [31]:
df_crz2025 = df_crz2025.reset_index()

In [33]:
df_crz2025['quarter'] = df_crz2025['quarter'].astype(str)

## Adding Inflation and RGDP Target

In [35]:

# 1. Define the targets (Midpoints used for ranges)
# Source: Government Work Reports
china_gdp_targets = {
    2000: 7.0, 2001: 7.0, 2002: 7.0, 2003: 7.0, 2004: 7.0,
    2005: 8.0, 2006: 8.0, 2007: 8.0, 2008: 8.0, 2009: 8.0,
    2010: 8.0, 2011: 8.0, 2012: 7.5, 2013: 7.5, 2014: 7.5,
    2015: 7.0, 
    2016: 6.75, # Range 6.5-7.0 -> Midpoint 6.75
    2017: 6.5, 
    2018: 6.5, 
    2019: 6.25, # Range 6.0-6.5 -> Midpoint 6.25
    2020: 6.0, # No official target set Set to 6.0 for calculation
    2021: 6.0,  # "Above 6.0" -> Treated as 6.0 baseline
    2022: 5.5, 
    2023: 5.0,
    2024: 5.0,
    2025: 5.0
}

china_cpi_targets = {
    2000: 2.0, 2001: 1.5, 2002: 1.5, 2003: 1.0, 2004: 3.0,
    2005: 4.0, 2006: 3.0, 2007: 3.0, 2008: 4.8, 2009: 4.0,
    2010: 3.0, 2011: 4.0, 2012: 4.0, 2013: 3.5, 2014: 3.5,
    2015: 3.0, 2016: 3.0, 2017: 3.0, 2018: 3.0, 2019: 3.0,
    2020: 3.5, 2021: 3.0, 2022: 3.0, 2023: 3.0, 2024: 3.0,
    2025: 2.0
}

years = df_crz2025['quarter'].str[:4].astype(int)
# 2. Extract the year from your PeriodIndex (assuming index is '2000Q1' etc.)
# If your index is named something else, change 'df_final.index'
df_crz2025['target_gdp'] = years.map(china_gdp_targets)
df_crz2025['target_cpi'] = years.map(china_cpi_targets)

# 4. Calculate the Gap immediately
df_crz2025['gdp_gap'] = df_crz2025['realgdpgrowth'] - df_crz2025['target_gdp']
df_crz2025['cpi_gap'] = df_crz2025['cpi'] - df_crz2025['target_cpi']


In [36]:
df_crz2025.to_csv('crz2025.csv', index=False)

## (Value-Added) Industrial Production

In [3]:
ip_df = ak.macro_china_industrial_production_yoy()

In [4]:
# Remove unnecessary columns and rename to English
ip_df = ip_df[["日期", "今值"]].rename(columns={"日期": "date", "今值": "ip"})


In [5]:
# Convert date into proper format
ip_df['date'] = pd.to_datetime(ip_df['date'])

In [6]:
# Remove values before 2000
ip_df = ip_df[ip_df['date'].dt.year >= 2000]

In [7]:
# Convert into quarter format
ip_df['quarter'] = ip_df['date'].dt.to_period('Q')

In [8]:
# Average within each quarter
ip_df = ip_df.groupby('quarter')['ip'].mean().reset_index()

## Composite Monetary Policy Index (CMPI)

In [13]:
# Get all sheet names from CMPI.xlsx
excel_file = pd.ExcelFile("CMPI.xlsx")
sheet_names = excel_file.sheet_names

# Dictionary to store all dataframes
cmpi_dfs = {}

# Loop through each sheet
for sheet_name in sheet_names:
    # 1. Read the sheet and name the df
    df = pd.read_excel("CMPI.xlsx", sheet_name=sheet_name)
    
    # 2. Remove the first row
    df = df.iloc[1:]
    
    # 3. Rename first column to 'date' and convert to datetime
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "date"})
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # 4. Remove values before 2000
    df = df[df['date'].dt.year >= 2000]
    
    # 5. Generate quarter variable based on date
    df['quarter'] = df['date'].dt.to_period('Q')
    
    # 6. Take average value for each quarter (for all numeric columns)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        df = df.groupby('quarter')[numeric_cols].mean().reset_index()
    
    # Store in dictionary with sheet name
    cmpi_dfs[f"{sheet_name}_df"] = df
    
    # Also create individual variable (optional)
    globals()[f"{sheet_name}_df"] = df

print(f"Processed {len(sheet_names)} sheets from CMPI.xlsx")
print("Available dataframes:", list(cmpi_dfs.keys()))

Processed 13 sheets from CMPI.xlsx
Available dataframes: ['正回购利率_df', '逆回购利率_df', '国库现金利率_df', '常备借贷便利利率_df', '短期流动性工具利率_df', '央行票据发行利率_df', '定期存款基准利率_df', '金融机构在央行存款利率_df', '中长期贷款基准利率_df', '中期借贷便利（MLF)利率_df', '抵押补充贷款利率_df', '贷款市场报价利率（LPR)_df', '存款准备金率（RRR）_df']


In [14]:
# Join all CMPI dataframes by quarter
cmpi_list = list(cmpi_dfs.values())
cmpi_df = reduce(lambda left, right: pd.merge(left, right, on='quarter', how='outer'), cmpi_list)

# Get all numeric columns (excluding quarter)
numeric_cols = cmpi_df.select_dtypes(include=[np.number]).columns.tolist()

# Standardize each numeric variable: (x - mean) / std
for col in numeric_cols:
    mean_val = cmpi_df[col].mean()
    std_val = cmpi_df[col].std()
    cmpi_df[f'{col}_standardized'] = (cmpi_df[col] - mean_val) / std_val

# Get all standardized columns
standardized_cols = [col for col in cmpi_df.columns if 'standardized' in col]

# Calculate CMPI as the average of all standardized variables for each quarter
cmpi_df['cmpi'] = cmpi_df[standardized_cols].mean(axis=1)

# Convert quarter to string for easier merging later
cmpi_df['quarter'] = cmpi_df['quarter'].astype(str)

print("CMPI dataframe created successfully with standardized values!")
print(f"Shape: {cmpi_df.shape}")
print(cmpi_df[['quarter', 'cmpi']].head())

MergeError: Passing 'suffixes' which cause duplicate columns {'date_x', 'Unnamed: 1_x', 'Unnamed: 2_x', 'Unnamed: 3_x'} is not allowed.